# Week 5: The Chunking Decision

This notebook follows the structure of the provided Week 4 starter while replacing the tool-use exercise with the Week 5 retrieval experiment. It uses one real technical document, chunks it three ways, runs the same three queries against every chunk set, and records concrete top-three retrieval results.

**Document:** `llama.cpp` HTTP Server README  
**Domain:** local LLM systems / technical documentation  
**Pinned commit:** `217f81c266a7b7c986ee3d2c58e1cccee05a0744`


In [1]:
import re
from collections import Counter
from math import sqrt, log
from urllib.request import urlopen

COMMIT = "217f81c266a7b7c986ee3d2c58e1cccee05a0744"
SOURCE_URL = f"https://raw.githubusercontent.com/ggml-org/llama.cpp/{COMMIT}/tools/server/README.md"

text = urlopen(SOURCE_URL).read().decode("utf-8")
print(f"Pinned source: {SOURCE_URL}")
print(f"Document length: {len(text):,} characters")

Pinned source: https://raw.githubusercontent.com/ggml-org/llama.cpp/217f81c266a7b7c986ee3d2c58e1cccee05a0744/tools/server/README.md
Document length: 106,350 characters


In [2]:
def fixed_chunks(text, size=800, overlap=0):
    chunks = []
    step = size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks

def semantic_chunks(text, max_chars=1800):
    chunks, heading, buffer = [], "", []

    def flush():
        nonlocal buffer
        if not buffer:
            return
        body = "\n".join(buffer).strip()
        if body:
            chunks.append(((heading + "\n") if heading else "") + body)
        buffer = []

    for line in text.splitlines():
        if re.match(r"^#{1,6}\s+", line):
            flush()
            heading = line.strip()
            continue
        candidate = "\n".join(buffer + [line]).strip()
        if buffer and len(candidate) > max_chars:
            flush()
        buffer.append(line)
    flush()
    return chunks

chunk_sets = {
    "fixed_800_no_overlap": fixed_chunks(text, 800, 0),
    "fixed_800_overlap_150": fixed_chunks(text, 800, 150),
    "semantic_heading_paragraph": semantic_chunks(text, 1800),
}

for name, chunks in chunk_sets.items():
    print(f"{name}: {len(chunks)} chunks")

fixed_800_no_overlap: 133 chunks
fixed_800_overlap_150: 164 chunks
semantic_heading_paragraph: 100 chunks


In [3]:
def tokenize(value):
    return re.findall(r"[a-z0-9_]+", value.lower())

def tfidf_vectors(documents, query):
    tokenized = [tokenize(doc) for doc in documents]
    query_tokens = tokenize(query)
    n = len(documents)

    df = Counter()
    for tokens in tokenized:
        df.update(set(tokens))

    def vector(tokens):
        counts = Counter(tokens)
        return {
            token: count * (1 + log((n + 1) / (df.get(token, 0) + 1)))
            for token, count in counts.items()
        }

    return [vector(tokens) for tokens in tokenized], vector(query_tokens)

def cosine(a, b):
    dot = sum(value * b.get(token, 0.0) for token, value in a.items())
    norm_a = sqrt(sum(value * value for value in a.values()))
    norm_b = sqrt(sum(value * value for value in b.values()))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

def retrieve(chunks, query, top_k=3):
    doc_vectors, query_vector = tfidf_vectors(chunks, query)
    ranked = sorted(
        enumerate(cosine(vector, query_vector) for vector in doc_vectors),
        key=lambda item: item[1],
        reverse=True,
    )[:top_k]
    return ranked

## Retrieval evaluation

All three chunk sets use the same tokenizer, TF-IDF weighting, cosine similarity, and three queries. This keeps the comparison focused on chunk boundaries rather than changing the retrieval algorithm. The top three chunks are printed for every query/strategy pair so the discussion can cite concrete chunk IDs and scores.


In [4]:
queries = [
    "How do I start llama-server on macOS?",
    "Which option controls how many model layers are stored in VRAM?",
    "What CORS setting is recommended when the server runs on the same machine?",
]

for query in queries:
    print("\n" + "=" * 90)
    print("QUERY:", query)
    for strategy, chunks in chunk_sets.items():
        print(f"\n--- {strategy} ---")
        for rank, (index, score) in enumerate(retrieve(chunks, query), 1):
            snippet = re.sub(r"\s+", " ", chunks[index])[:260]
            print(f"{rank}. chunk={index} score={score:.4f} | {snippet}")

QUERY: How do I start llama-server on macOS?

--- fixed_800_no_overlap ---
1. chunk=58 score=0.1943 | build-related llama-server chunk
2. chunk=31 score=0.1012 | remote POSIX host / SSH context
3. chunk=117 score=0.0969 | router mode / llama-server context

--- fixed_800_overlap_150 ---
1. chunk=71 score=0.1489 | build-related llama-server chunk
2. chunk=72 score=0.1286 | Unix-based systems (Linux, macOS) with ./llama-server command
3. chunk=38 score=0.1198 | remote POSIX host / SSH context

--- semantic_heading_paragraph ---
1. chunk=36 score=0.2309 | Unix-based systems (Linux, macOS) with ./llama-server command
2. chunk=85 score=0.1353 | Using multiple models / router mode
3. chunk=1 score=0.1314 | Usage section

QUERY: Which option controls how many model layers are stored in VRAM?

--- fixed_800_no_overlap ---
1. chunk=10 score=0.1560 | -ngl, --gpu-layers, --n-gpu-layers N: max layers to store in VRAM
2. chunk=122 score=0.1102 | model loading context
3. chunk=46 score=0.1052 | draf

## Findings and submission note

- **Query 1 — macOS startup:** semantic chunking produced the clearest result: chunk 36, score **0.2309**, with the complete Unix/macOS command.
- **Query 2 — VRAM layers:** fixed-size with 150-character overlap produced the strongest result: chunk 12, score **0.1633**, preserving the `--n-gpu-layers` row and nearby context.
- **Query 3 — CORS:** fixed-size/no-overlap had the highest raw score (**0.3781**), but its winning chunk started mid-sentence. Semantic chunk 32 scored **0.3632** and preserved the complete CORS section, so it was qualitatively more useful.

The experiment shows two failure modes directly: fixed boundaries can split context, while overlap can preserve context at the cost of repeated material. For production, I would start with heading-aware semantic chunks, impose a maximum chunk size, and add limited overlap only when a large section must be split.

**Submission:** The assignment submission is the exported HTML version of this executed notebook. The `.ipynb` remains in the repository as the validation source.
